# 📄 Chat with a PDF — RAG Pipeline from Scratch

**Retrieval-Augmented Generation (RAG)** is one of the most common real-world AI engineering tasks.  
This notebook builds a complete RAG pipeline from scratch — no magic, no black boxes.

### What we build
Given any PDF, we can ask it questions and get grounded answers.

### The pipeline (6 steps)
```
PDF URL → Load → Chunk → Embed → FAISS Index → Retrieve → Groq LLM → Answer
```

### Stack
| Component | Tool | Why |
|---|---|---|
| PDF loading | `pypdf` | Extracts raw text from any PDF |
| Chunking | `langchain-text-splitters` | Splits text into overlapping windows |
| Embeddings | `sentence-transformers` (HuggingFace) | Free, runs locally, no API key needed |
| Vector DB | `FAISS` | Fast similarity search by Facebook AI |
| LLM | `Groq` (llama-3.3-70b) | Free API, extremely fast inference |

---
**Paper used:** [Attention Is All You Need](https://arxiv.org/pdf/1706.03762) — Vaswani et al., 2017

## Step 1 — Install Dependencies

In [1]:
# Install all required packages
# - langchain + langchain-community: framework that connects all components
# - langchain-huggingface: lets LangChain use HuggingFace embedding models
# - langchain-text-splitters: text chunking utilities
# - faiss-cpu: Facebook's vector similarity search library
# - sentence-transformers: the actual embedding model (all-MiniLM-L6-v2)
# - pypdf: PDF text extraction
# - groq: Groq API client for the LLM

!pip install -q \
    langchain \
    langchain-community \
    langchain-huggingface \
    langchain-text-splitters \
    faiss-cpu \
    sentence-transformers \
    pypdf \
    groq

print("All packages installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 45.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 66.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2,

## Step 2 — Load the PDF

We download the PDF into memory and extract raw text from all pages.  
No file is saved to disk — we read it directly from the URL.

In [2]:
import urllib.request
from pypdf import PdfReader
import io

# --- Config: swap this URL to use any other PDF ---
PDF_URL = "https://arxiv.org/pdf/1706.03762"

# Download the PDF bytes into memory
req = urllib.request.Request(PDF_URL, headers={"User-Agent": "Mozilla/5.0"})
pdf_bytes = urllib.request.urlopen(req).read()

# Extract text from every page
reader = PdfReader(io.BytesIO(pdf_bytes))
pages = [page.extract_text() for page in reader.pages]
full_text = "\n".join(pages)

print(f"Pages loaded   : {len(pages)}")
print(f"Total chars    : {len(full_text):,}")
print(f"\n--- First 400 characters ---")
print(full_text[:400])

Pages loaded   : 15
Total chars    : 39,629

--- First 400 characters ---
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
G


## Step 3 — Chunk the Text

LLMs have a context limit — we can't feed the whole PDF at once.  
We split it into small overlapping chunks so no sentence is cut off at a boundary.

**Key parameters:**
- `chunk_size=500` — max characters per chunk
- `chunk_overlap=50` — how many characters to repeat between adjacent chunks

The overlap ensures a sentence split across two chunks isn't lost.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,        # max characters per chunk
    chunk_overlap=50,      # overlap to preserve context at boundaries
    separators=["\n\n", "\n", " ", ""]  # tries paragraph breaks first
)

chunks = splitter.split_text(full_text)

print(f"Total chunks   : {len(chunks)}")
print(f"Avg chunk size : {sum(len(c) for c in chunks) // len(chunks)} characters")

# Show the overlap in action
print(f"\n--- End of chunk 0 ---")
print(repr(chunks[0][-80:]))
print(f"\n--- Start of chunk 1 ---")
print(repr(chunks[1][:80]))
print("\n↑ Same text appears at the end of chunk 0 and start of chunk 1 — that's the overlap.")

Total chunks   : 89
Avg chunk size : 455 characters

--- End of chunk 0 ---
'ch\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu'

--- Start of chunk 1 ---
'University of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser ∗\nGoogle Brain\nlukaszka'

↑ Same text appears at the end of chunk 0 and start of chunk 1 — that's the overlap.


## Step 4 — Embed the Chunks

We convert each chunk into a **vector** (a list of numbers) that represents its meaning.  
Chunks with similar meaning will have similar vectors — this is what enables semantic search.

**Model:** `all-MiniLM-L6-v2` — a small, fast HuggingFace model that produces 384-dimensional vectors.  
Downloads ~90MB on first run, then cached.

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

# Load the embedding model (downloads on first run, cached after)
embedder = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Embed a single chunk to inspect what a vector looks like
sample_vector = embedder.embed_query(chunks[0])

print(f"Embedding model : all-MiniLM-L6-v2")
print(f"Vector size     : {len(sample_vector)} dimensions")
print(f"\nFirst 10 values of chunk 0's vector:")
print([round(v, 4) for v in sample_vector[:10]])
print("\nThose 384 numbers encode the *meaning* of the text.")
print("Similar text → similar numbers → found by similarity search.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model : all-MiniLM-L6-v2
Vector size     : 384 dimensions

First 10 values of chunk 0's vector:
[0.1132, -0.0085, 0.0096, 0.0503, 0.008, -0.0028, -0.0367, -0.001, -0.0289, 0.1407]

Those 384 numbers encode the *meaning* of the text.
Similar text → similar numbers → found by similarity search.


## Step 5 — Build the FAISS Vector Index

FAISS (Facebook AI Similarity Search) stores all 89 vectors and lets us search them instantly.  
When we embed a query, FAISS finds the chunks whose vectors are closest to the query vector.

This is **cosine similarity** — not keyword matching.

In [5]:
from langchain_community.vectorstores import FAISS

# Embed all chunks and build the searchable index
print("Building FAISS index...")
db = FAISS.from_texts(chunks, embedder)
print(f"Index built! Vectors stored: {db.index.ntotal}")

# --- Test retrieval immediately ---
test_query = "What is the attention mechanism?"
test_results = db.similarity_search(test_query, k=3)

print(f"\n--- Test: top 3 chunks for '{test_query}' ---")
for i, doc in enumerate(test_results):
    print(f"\n[{i+1}] {doc.page_content[:200]}...")

Building FAISS index...
Index built! Vectors stored: 89

--- Test: top 3 chunks for 'What is the attention mechanism?' ---

[1] reduced to a constant number of operations, albeit at the cost of reduced effective resolution due
to averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as
descr...

[2] from our models and present and discuss examples in the appendix. Not only do individual attention
heads clearly learn to perform different tasks, many appear to exhibit behavior related to the syntac...

[3] 3.2.3 Applications of Attention in our Model
The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder laye...


## Step 6 — Load Groq API Key

We load the key securely from Kaggle Secrets — never hardcode API keys in notebooks.

**To add your key:**  
Right panel → Add-ons → Secrets → Add `GROQ_API_KEY` → toggle on.  
Get a free key at [console.groq.com](https://console.groq.com).

In [7]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
GROQ_API_KEY = secrets.get_secret("GROQ_API_KEY")

print("Groq API key loaded successfully!")

Groq API key loaded successfully!


## Step 7 — Ask the PDF a Question

The full RAG loop in one cell:
1. Embed the query
2. Retrieve top-k most relevant chunks from FAISS
3. Build a prompt with those chunks as context
4. Send to Groq LLM
5. Return grounded answer

**Change `query` to anything and re-run.**

In [8]:
from groq import Groq

def ask_pdf(query, k=5):
    """
    Full RAG pipeline:
      query -> retrieve top-k chunks -> build prompt -> LLM -> answer
    
    Args:
        query: the question to ask
        k: number of chunks to retrieve (more = richer context, higher token cost)
    """
    # Step 1: retrieve relevant chunks via vector similarity
    results = db.similarity_search(query, k=k)
    context = "\n\n".join(doc.page_content for doc in results)

    # Step 2: build a grounded prompt
    # "ONLY using the context" is critical — prevents hallucination
    system_prompt = (
        "You are a helpful assistant that answers questions about a research paper.\n"
        "Answer ONLY using the context provided below. "
        "If the answer is not in the context, say 'I don\'t find that in the paper.' "
        "Do not make anything up.\n\n"
        f"Context:\n{context}"
    )

    # Step 3: send to Groq
    client = Groq(api_key=GROQ_API_KEY)
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": query}
        ]
    )

    answer = response.choices[0].message.content

    # Print everything for full transparency
    print("=" * 60)
    print(f"QUERY  : {query}")
    print("=" * 60)
    print(f"CONTEXT (top {k} chunks, first 600 chars):")
    print(context[:600], "...")
    print("=" * 60)
    print("ANSWER :")
    print(answer)
    print("=" * 60)
    return answer


# --- Try different questions below ---
ask_pdf("How does multi-head attention work?")

QUERY  : How does multi-head attention work?
CONTEXT (top 5 chunks, first 600 chars):
reduced to a constant number of operations, albeit at the cost of reduced effective resolution due
to averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as
described in section 3.2.
Self-attention, sometimes called intra-attention is an attention mechanism relating different positions
of a single sequence in order to compute a representation of the sequence. Self-attention has been

3.2.3 Applications of Attention in our Model
The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from  ...
ANSWER :
Multi-head attention is described in the context as a mechanism that uses multiple parallel attention layers, or heads. In this work, they employ h = 8 parallel attention layers. For each of these, they use dk = dv = dmodel/h = 64, which reduces the dimension of each head. The projections are parame

'Multi-head attention is described in the context as a mechanism that uses multiple parallel attention layers, or heads. In this work, they employ h = 8 parallel attention layers. For each of these, they use dk = dv = dmodel/h = 64, which reduces the dimension of each head. The projections are parameter matrices WQ, WK, WV, and WO. The total computational cost is similar to that of single-head attention with full dimensionality. However, the exact details of how multi-head attention works are not fully explained in the provided context.'

In [9]:
# Try more questions — change and re-run this cell
ask_pdf("What BLEU score did the Transformer achieve on WMT 2014 English-to-German?")

QUERY  : What BLEU score did the Transformer achieve on WMT 2014 English-to-German?
CONTEXT (top 5 chunks, first 600 chars):
6 Results
6.1 Machine Translation
On the WMT 2014 English-to-German translation task, the big transformer model (Transformer (big)
in Table 2) outperforms the best previously reported models (including ensembles) by more than 2.0
BLEU, establishing a new state-of-the-art BLEU score of 28.4. The configuration of this model is
listed in the bottom line of Table 3. Training took 3.5 days on 8 P100 GPUs. Even our base model

warmup_steps = 4000.
5.4 Regularization
We employ three types of regularization during training:
7
Table 2: The Transformer achieves better BLEU scores than previous state-of- ...
ANSWER :
The Transformer (big) model achieved a BLEU score of 28.4 on the WMT 2014 English-to-German translation task.


'The Transformer (big) model achieved a BLEU score of 28.4 on the WMT 2014 English-to-German translation task.'

In [10]:
ask_pdf("What optimizer and learning rate schedule did they use?")

QUERY  : What optimizer and learning rate schedule did they use?
CONTEXT (top 5 chunks, first 600 chars):
(3.5 days).
5.3 Optimizer
We used the Adam optimizer [20] with β1 = 0.9, β2 = 0.98 and ϵ = 10 −9. We varied the learning
rate over the course of training, according to the formula:
lrate = d−0.5
model · min(step_num−0.5, step_num · warmup_steps−1.5) (3)
This corresponds to increasing the learning rate linearly for the first warmup_steps training steps,
and decreasing it thereafter proportionally to the inverse square root of the step number. We used
warmup_steps = 4000.
5.4 Regularization

target tokens.
5.2 Hardware and Schedule
We trained our models on one machine with 8 NVIDIA P100 GPUs. Fo ...
ANSWER :
They used the Adam optimizer with β1 = 0.9, β2 = 0.98, and ϵ = 10−9. The learning rate schedule was varied according to the formula: 

lrate = d−0.5 model · min(step_num−0.5, step_num · warmup_steps−1.5) 

This corresponds to increasing the learning rate linearly for the first wa

'They used the Adam optimizer with β1 = 0.9, β2 = 0.98, and ϵ = 10−9. The learning rate schedule was varied according to the formula: \n\nlrate = d−0.5 model · min(step_num−0.5, step_num · warmup_steps−1.5) \n\nThis corresponds to increasing the learning rate linearly for the first warmup_steps (which was set to 4000) training steps, and decreasing it thereafter proportionally to the inverse square root of the step number.'

In [11]:
ask_pdf("Why did the authors remove recurrence from the architecture?")

QUERY  : Why did the authors remove recurrence from the architecture?
CONTEXT (top 5 chunks, first 600 chars):
(x1, ..., xn) to another sequence of equal length (z1, ..., zn), with xi, zi ∈ Rd, such as a hidden
layer in a typical sequence transduction encoder or decoder. Motivating our use of self-attention we
consider three desiderata.
One is the total computational complexity per layer. Another is the amount of computation that can
be parallelized, as measured by the minimum number of sequential operations required.

Attention mechanisms have become an integral part of compelling sequence modeling and transduc-
tion models in various tasks, allowing modeling of dependencies without regard to their di ...
ANSWER :
The context does not explicitly state why the authors removed recurrence from the architecture, but it mentions that the proposed model, the Transformer, "eschewing recurrence" and that recurrent models have limitations, such as requiring O(n) sequential operations. This sug

'The context does not explicitly state why the authors removed recurrence from the architecture, but it mentions that the proposed model, the Transformer, "eschewing recurrence" and that recurrent models have limitations, such as requiring O(n) sequential operations. This suggests that the authors may have removed recurrence to improve the efficiency and parallelization of the model.'

## What We Learned

### RAG works because of semantic search
When we search for *"side effects"*, it also finds chunks about *"adverse reactions"* — same meaning, different words. Keyword search would miss that.

### Retrieval quality controls answer quality
The LLM can only answer as well as what we give it. If the right chunk isn't retrieved, the answer will be weak or "I don't find that." This is expected and honest behavior — not a bug.

### Tuning levers
| Parameter | Effect |
|---|---|
| `chunk_size` | Smaller = more precise retrieval. Larger = more context per chunk. |
| `chunk_overlap` | Higher = fewer boundary misses, more storage. |
| `k` (top-k) | Higher = richer context, but more tokens sent to LLM. |
| `model` | Larger models = better reasoning over complex context. |

### Common interview follow-ups
- *How would you evaluate retrieval quality?* → RAGAS, recall@k, MRR
- *What if the answer spans multiple chunks?* → Increase k, or use a re-ranker (Cohere Rerank)
- *How would you deploy this?* → FastAPI endpoint, FAISS index saved to disk, reload at startup
- *How do you handle PDFs with images/tables?* → OCR (pytesseract), or multimodal models

---
*Built with LangChain · HuggingFace · FAISS · Groq*